# EpidemicWatch — Phase 2 (LARGE DATA): Data Preparation

Replaces the 1,153-record Symptom2Disease data with a much larger dataset, while
producing **exactly the same output folder structure and column names**
(`text`, `label`, `syndrome_category`) so the later phases keep working.

**Source:** Kaggle "Diseases and Symptoms Dataset" (~246,000 rows, ~773 diseases,
~377 binary symptom columns). It stores symptoms as 0/1 columns, not text, so this
notebook turns each row into a natural-language sentence.

**Read these limitations before using the results in a report:**
1. **The text is template-generated.** Sentences are built from a row's symptom
   list using varied templates. Real patient text is far messier, so scores here
   will be higher than real-world performance. Report this as a synthetic-text benchmark.
2. **Syndrome labels come from keyword rules on disease names** (not clinician review).
   The mapping is saved to `mapping_review.csv` and printed so you can inspect and edit it.
3. **Duplicate/ambiguous symptom combinations are removed before splitting**, otherwise
   the same symptom set would appear in both train and test and inflate accuracy.

> CPU runtime is fine for this notebook.

## 0. Setup

In [ ]:
!pip install -q kagglehub

import os, re, json, glob, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng_global = np.random.default_rng(SEED)

# ---- Config -------------------------------------------------------------
N_HOSPITALS = 3
DIRICHLET_ALPHA = 0.8            # skew between hospitals (lower = more skewed)
MAX_PER_CATEGORY = 3000          # cap per syndrome category -> ~15,000 records total
MIN_CELL_REQUIRED = 100          # every (hospital, category) cell must have at least this many
MAX_SYMPTOMS_IN_TEXT = 8         # keeps sentences short enough for 64-token inputs
DISTRACTOR_PROB = 0.15           # chance of adding one unrelated symptom (realistic noise)
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
OUTPUT_DIR = "epidemicwatch_data"   # same name the later phases expect
KAGGLE_SLUG = "dhivyeshrk/diseases-and-symptoms-dataset"
CATEGORIES = ["Respiratory", "Gastrointestinal", "Febrile/Systemic", "Dermatological", "Musculoskeletal/Other"]

if os.path.isdir(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)   # start clean so old small-dataset files can't mix in
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config set.")

## 1. Load the Dataset

In [ ]:
csv_path = None
try:
    import kagglehub
    folder = kagglehub.dataset_download(KAGGLE_SLUG)
    candidates = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
    assert candidates, "no CSV found in downloaded folder"
    csv_path = max(candidates, key=os.path.getsize)
    print("Downloaded via kagglehub:", csv_path)
except Exception as e:
    print("kagglehub failed or not configured:", e)
    from google.colab import files
    print("Upload the dataset CSV downloaded from Kaggle.")
    up = files.upload()
    csv_path = list(up.keys())[0]

header = pd.read_csv(csv_path, nrows=0).columns.tolist()
print("Number of columns:", len(header))
print("First 8 columns:", header[:8])

label_candidates = [c for c in header if c.strip().lower() in ("diseases", "disease", "label", "diagnosis")]
LABEL_COL = label_candidates[0] if label_candidates else header[0]
SYMPTOM_COLS = [c for c in header if c != LABEL_COL]
print("Using label column:", repr(LABEL_COL), "| symptom columns:", len(SYMPTOM_COLS))

df = pd.read_csv(csv_path, dtype={c: "int8" for c in SYMPTOM_COLS})
vals = pd.unique(df[SYMPTOM_COLS].values.ravel()[:2_000_000])
assert set(vals).issubset({0, 1}), f"Symptom columns are not binary 0/1 (found {vals[:10]}). Dataset schema differs from expected."
assert len(SYMPTOM_COLS) >= 50, "Expected many symptom columns; schema differs from expected."
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
print(f"Loaded {len(df):,} rows, {df[LABEL_COL].nunique()} distinct diseases.")

## 2. Map Diseases to the 5 Syndrome Categories

Keyword rules on the disease name, checked in priority order. Anything that matches
no rule goes to `Musculoskeletal/Other` (the catch-all, same name as in the earlier
phases). **Review the printed examples** and edit the patterns if something is misplaced.

In [ ]:
RULES = [
    ("Dermatological", r"dermat|eczema|psoria|acne|rosacea|rash|skin|cellulitis|impetigo|wart|fungal|tinea|urticaria|hives|pruritus|scabies|lice|melanoma|carbuncle|abscess|boil|pressure ulcer|foot ulcer|cutaneous|herpes zoster|shingles|lichen|vitiligo|alopecia|keratosis|callus|sunburn"),
    ("Febrile/Systemic", r"fever|malaria|dengue|typhoid|sepsis|septic|mononucleosis|lyme|\bhiv\b|\baids\b|meningitis|encephalitis|brucell|leptospir|rickett|chikungunya|ebola|zika|cholera|measles|varicella|chickenpox|rubella|mumps|scarlet|toxic shock|viral infection|bacteremia|cytomegalo|infectious"),
    ("Respiratory", r"pneumon|bronch|asthma|copd|emphysema|pulmon|pleur|tubercul|influenza|\bflu\b|common cold|sinusitis|pharyngitis|laryng|tonsill|croup|whooping|pertussis|respiratory|lung|cough|sleep apnea|rhinitis|nasal"),
    ("Gastrointestinal", r"gastr|ulcer|colitis|crohn|appendic|cholecyst|gallbladder|gallstone|pancreat|diarrhea|hernia|bowel|esophag|diverticul|celiac|intestin|constipation|hemorrhoid|rectal|hepatitis|cirrhosis|liver|jaundice|dysphagia|ileus|volvulus|stomach|peptic|reflux|cholang|duoden|colon|norovirus|rotavirus|food poisoning"),
]
CATCH_ALL = "Musculoskeletal/Other"

def map_disease(name):
    for cat, pat in RULES:
        if re.search(pat, name):
            return cat
    return CATCH_ALL

disease_table = (df[LABEL_COL].value_counts().rename_axis("disease").reset_index(name="rows"))
disease_table["syndrome_category"] = disease_table["disease"].map(map_disease)
disease_table.to_csv(os.path.join(OUTPUT_DIR, "mapping_review.csv"), index=False)

df["syndrome_category"] = df[LABEL_COL].map(dict(zip(disease_table["disease"], disease_table["syndrome_category"])))

print("Rows and distinct diseases per category (BEFORE dedup/sampling):")
summary = df.groupby("syndrome_category").agg(rows=(LABEL_COL, "size"), diseases=(LABEL_COL, "nunique"))
print(summary.reindex(CATEGORIES).to_string())
print()
for cat in CATEGORIES:
    ex = disease_table[disease_table["syndrome_category"] == cat].head(8)["disease"].tolist()
    print(f"{cat}: {ex}")

## 3. Remove Duplicate and Ambiguous Symptom Combinations

Many rows share an identical symptom set. If they stay, the same combination lands in
both train and test (leakage). We keep one row per distinct symptom set, and drop any
symptom set that maps to more than one syndrome (ambiguous).

In [ ]:
packed = np.packbits(df[SYMPTOM_COLS].values.astype(np.uint8), axis=1)
df["_key"] = [p.tobytes() for p in packed]
n0 = len(df)

n_syndromes_per_key = df.groupby("_key")["syndrome_category"].nunique()
ambiguous_keys = set(n_syndromes_per_key[n_syndromes_per_key > 1].index)
df = df[~df["_key"].isin(ambiguous_keys)]
n1 = len(df)
df = df.drop_duplicates(subset="_key", keep="first").reset_index(drop=True)
n2 = len(df)
print(f"Rows: {n0:,} -> {n1:,} after dropping ambiguous symptom sets -> {n2:,} after removing duplicates")

df["n_symptoms"] = df[SYMPTOM_COLS].sum(axis=1)
df = df[df["n_symptoms"] >= 1].reset_index(drop=True)
print("Category sizes after cleaning:")
print(df["syndrome_category"].value_counts().reindex(CATEGORIES).to_string())
too_small = [c for c in CATEGORIES if (df["syndrome_category"] == c).sum() < 500]
if too_small:
    print("\n!! WARNING: these categories have fewer than 500 usable rows:", too_small,
          "-- consider loosening the keyword rules above.")

## 4. Balanced Sample (cap per category)

In [ ]:
parts = []
for cat in CATEGORIES:
    sub = df[df["syndrome_category"] == cat]
    n = min(len(sub), MAX_PER_CATEGORY)
    parts.append(sub.sample(n=n, random_state=SEED))
sample_df = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"Sample size: {len(sample_df):,}")
print(sample_df["syndrome_category"].value_counts().reindex(CATEGORIES).to_string())

## 5. Turn Symptom Lists into Natural-Language Text

Each row's symptoms are shuffled, optionally mixed with one unrelated "distractor" symptom,
and placed into one of several sentence templates.

In [ ]:
TEMPLATES = [
    "I have {s}.",
    "For the past few days I've had {s}.",
    "Patient reports {s}.",
    "I've been experiencing {s} and I'm worried.",
    "Complaining of {s}.",
    "My main problems are {s}.",
    "Chief complaint: {s}.",
    "Since yesterday I've noticed {s}.",
    "I feel unwell, with {s}.",
    "Lately I keep having {s}.",
    "I came in because of {s}.",
    "The symptoms are {s}.",
]
DURATIONS = ["", "", "", " It started two days ago.", " This has lasted about a week.",
             " It got worse overnight.", " It began this morning.", " It comes and goes."]

def join_list(items):
    if len(items) == 1: return items[0]
    return ", ".join(items[:-1]) + " and " + items[-1]

symptom_names = [c.replace("_", " ").strip().lower() for c in SYMPTOM_COLS]
sym_freq = df[SYMPTOM_COLS].mean().values
sym_prob = sym_freq / sym_freq.sum()
sym_matrix = sample_df[SYMPTOM_COLS].values

text_rng = np.random.default_rng(SEED + 7)
texts = []
for row in sym_matrix:
    idx = np.flatnonzero(row)
    text_rng.shuffle(idx)
    idx = list(idx[:MAX_SYMPTOMS_IN_TEXT])
    names = [symptom_names[i] for i in idx]
    if text_rng.random() < DISTRACTOR_PROB:
        extra = symptom_names[text_rng.choice(len(symptom_names), p=sym_prob)]
        if extra not in names:
            names.insert(text_rng.integers(0, len(names) + 1), extra)
    sentence = TEMPLATES[text_rng.integers(len(TEMPLATES))].format(s=join_list(names))
    sentence += DURATIONS[text_rng.integers(len(DURATIONS))]
    texts.append(sentence)

final_df = pd.DataFrame({"text": texts, "label": sample_df[LABEL_COL].values,
                         "syndrome_category": sample_df["syndrome_category"].values})
final_df = final_df.drop_duplicates(subset="text").reset_index(drop=True)
print(f"{len(final_df):,} records after removing identical sentences.")
for i in range(5):
    print(f"[{final_df.syndrome_category[i]} | {final_df.label[i]}] {final_df.text[i]}")
print("Average words per text:", round(final_df["text"].str.split().str.len().mean(), 1))

## 6. Non-IID and IID Hospital Splits

Same method as the small-data version (Dirichlet per category). With much more data
this notebook searches seeds until every (hospital, category) cell has at least
`MIN_CELL_REQUIRED` records, so no hospital ends up missing a category.

In [ ]:
def dirichlet_non_iid_split(df, n_hospitals, alpha, seed):
    rng = np.random.default_rng(seed)
    idx_by_h = {h: [] for h in range(n_hospitals)}
    for cat in CATEGORIES:
        cat_idx = df.index[df["syndrome_category"] == cat].to_numpy().copy()
        rng.shuffle(cat_idx)
        props = rng.dirichlet([alpha] * n_hospitals)
        counts = (props * len(cat_idx)).astype(int)
        for i in range(len(cat_idx) - counts.sum()):
            counts[i % n_hospitals] += 1
        start = 0
        for h in range(n_hospitals):
            idx_by_h[h].extend(cat_idx[start:start + counts[h]].tolist())
            start += counts[h]
    return {h: df.loc[ix].reset_index(drop=True) for h, ix in idx_by_h.items()}

def iid_split(df, n_hospitals, seed):
    rng = np.random.default_rng(seed)
    idx_by_h = {h: [] for h in range(n_hospitals)}
    for cat in CATEGORIES:
        cat_idx = df.index[df["syndrome_category"] == cat].to_numpy().copy()
        rng.shuffle(cat_idx)
        for h, chunk in enumerate(np.array_split(cat_idx, n_hospitals)):
            idx_by_h[h].extend(chunk.tolist())
    return {h: df.loc[ix].reset_index(drop=True) for h, ix in idx_by_h.items()}

def min_cell(shards):
    return min(int((s["syndrome_category"] == c).sum()) for s in shards.values() for c in CATEGORIES)

chosen_seed, non_iid_shards = None, None
for seed in range(SEED, SEED + 200):
    candidate = dirichlet_non_iid_split(final_df, N_HOSPITALS, DIRICHLET_ALPHA, seed)
    if min_cell(candidate) >= MIN_CELL_REQUIRED:
        chosen_seed, non_iid_shards = seed, candidate
        break
if non_iid_shards is None:
    raise RuntimeError("No seed met MIN_CELL_REQUIRED. Raise DIRICHLET_ALPHA or lower MIN_CELL_REQUIRED.")
iid_shards = iid_split(final_df, N_HOSPITALS, chosen_seed)
print(f"Chosen seed: {chosen_seed} | smallest (hospital, category) cell: {min_cell(non_iid_shards)}")
for h, s in non_iid_shards.items():
    print(f"Hospital {h}: {len(s):,} records ->", s["syndrome_category"].value_counts().reindex(CATEGORIES).to_dict())

## 7. Visualize

In [ ]:
def plot_dist(shards, title, fname):
    hs = sorted(shards)
    mat = np.array([[ (shards[h]["syndrome_category"] == c).sum() for c in CATEGORIES] for h in hs])
    fig, ax = plt.subplots(figsize=(9, 5)); bottom = np.zeros(len(hs))
    for i, c in enumerate(CATEGORIES):
        ax.bar([f"Hospital {h}" for h in hs], mat[:, i], bottom=bottom, label=c, color=plt.cm.tab10.colors[i]); bottom += mat[:, i]
    ax.set_title(title); ax.set_ylabel("Records"); ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150); plt.show()

plot_dist(non_iid_shards, "Category mix per hospital (non-IID)", "non_iid_distribution.png")
plot_dist(iid_shards, "Category mix per hospital (IID control)", "iid_distribution.png")

## 8. Train/Val/Test Split Within Each Hospital, Save Everything

In [ ]:
from sklearn.model_selection import train_test_split

def split_shard(shard, seed=SEED):
    train, temp = train_test_split(shard, train_size=TRAIN_FRAC, random_state=seed, stratify=shard["syndrome_category"])
    val, test = train_test_split(temp, train_size=VAL_FRAC / (VAL_FRAC + TEST_FRAC), random_state=seed, stratify=temp["syndrome_category"])
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

def save_all(shards, name):
    manifest = {"split_type": name, "n_hospitals": len(shards),
                "dirichlet_alpha": DIRICHLET_ALPHA if name == "non_iid" else None, "hospitals": {}}
    for h, shard in shards.items():
        tr, va, te = split_shard(shard)
        d = os.path.join(OUTPUT_DIR, name, f"hospital_{h}")
        os.makedirs(d, exist_ok=True)
        for part, frame in (("train", tr), ("val", va), ("test", te)):
            frame.to_csv(os.path.join(d, f"{part}.csv"), index=False)
        manifest["hospitals"][f"hospital_{h}"] = {
            "total_size": len(shard), "train_size": len(tr), "val_size": len(va), "test_size": len(te),
            "category_distribution": shard["syndrome_category"].value_counts().to_dict()}
    with open(os.path.join(OUTPUT_DIR, f"manifest_{name}.json"), "w") as f:
        json.dump(manifest, f, indent=2)
    return manifest

m_non = save_all(non_iid_shards, "non_iid")
m_iid = save_all(iid_shards, "iid")
for h, info in m_non["hospitals"].items():
    print(h, {k: v for k, v in info.items() if k.endswith("size")})

print("\nSUMMARY")
print(f"Total records: {len(final_df):,} | distinct diseases: {final_df['label'].nunique()}")
print(f"Vocabulary size (unique words): {len(set(' '.join(final_df['text'].str.lower()).split())):,}")
print(f"Avg text length (words): {final_df['text'].str.split().str.len().mean():.1f}")

## 9. Zip for Download (upload this zip into the later phases)

In [ ]:
zip_path = shutil.make_archive("epidemicwatch_data", "zip", OUTPUT_DIR)
print("Created:", zip_path)
# from google.colab import files; files.download(zip_path)